# 08 – M7: LLM Qwen2.5-7B-Instruct phân loại bằng prompt (W4-12)

**Người chạy: Quang.** Chạy lần lượt từ trên xuống, ô trước xong mới chạy ô sau. Tổng khoảng **60–90 phút** GPU.

**Bắt buộc dùng GPU:** Runtime → Change runtime type → **T4 GPU** → Save.

M7 không huấn luyện gì: đưa cặp câu vào prompt hỏi trắc nghiệm A/B/C rồi lấy chữ có xác suất cao nhất.
- **S0 – zero-shot:** chỉ có hướng dẫn, chạy trên XNLI, ViANLI, ViNLI test;
- **FS – few-shot:** thêm 6 ví dụ có nhãn từ ViANLI train (2 ví dụ mỗi nhãn), 3 bộ ví dụ khác nhau (seed 42, 123, 2024).

Lần đầu tải mô hình khoảng 15 GB (5–10 phút). Kết quả tự lưu vào Drive chung `NLI/results/`. Không cần file tách từ.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results', 'checkpoints']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

## Kiểm tra GPU, dữ liệu, thư viện 4-bit

In [ ]:
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU, bật T4 GPU rồi chạy lại từ ô đầu')
need = [f'data/processed/{f}.jsonl' for f in ['vianli_train', 'vianli_test', 'vinli_test', 'xnli_test']]
missing = [f for f in need if not os.path.exists(f)]
print('Thiếu file dữ liệu:', missing if missing else 'không, đủ cả 4 file')
import bitsandbytes, transformers
print('bitsandbytes', bitsandbytes.__version__, '| transformers', transformers.__version__)
print('Phiên bản thư viện (cho báo cáo mục 4.1):')
import sklearn, gensim
print('torch', torch.__version__, '| transformers', transformers.__version__, '| sklearn', sklearn.__version__, '| gensim', gensim.__version__)

**Đạt khi:** dòng GPU ghi `Tesla T4`, không thiếu file, in được phiên bản bitsandbytes. **Chụp lại dòng "Phiên bản thư viện" gửi Quân** (để điền báo cáo mục 4.1).

## Bước 1: chạy thử 30 cặp mỗi tập (khoảng 10 phút, phần lớn là tải mô hình)

In [ ]:
!python scripts/run_llm.py --mode zero few --smoke

In [ ]:
!python scripts/check_llm.py --smoke

**Đạt khi:** có 5 dòng (S0 × 3 tập, FS × 2 tập), `4-bit=True`, thiết bị `cuda:0` và dòng cuối `OK, chạy tiếp bước 2`.
Nếu ô chạy thử ở trên đã in đủ 5 dòng `[M7_...] accuracy=...` là đạt.
Accuracy chạy thử dao động mạnh vì chỉ 30 cặp, không cần quan tâm. Nếu một dòng **dự đoán toàn một nhãn** (ví dụ `{'entailment': 30, ...: 0}`)
thì vẫn chạy tiếp nhưng nhắn Quân.

**Nếu báo lỗi hết bộ nhớ (CUDA out of memory):** thêm ` --batch-size 2` vào cuối các lệnh bên dưới.

## Bước 2: S0 – zero-shot trên 3 tập test (khoảng 20–30 phút)

In [ ]:
!python scripts/run_llm.py --mode zero

## Bước 3: FS – few-shot, 3 bộ ví dụ (khoảng 30–50 phút)
Có thể chạy riêng ô này (sau khi chạy ô đầu) mà không cần chạy lại bước 2.

In [ ]:
!python scripts/run_llm.py --mode few --seeds 42 123 2024

## Bước 4: bảng tổng hợp và hình (vài giây)

In [ ]:
!python scripts/make_table.py
!python scripts/plot_results.py

In [ ]:
!python scripts/check_llm.py

## Lưu và nộp
- Kết quả đã nằm trên Drive chung `NLI/results/`. Ô trên phải in `Đủ 9 file kết quả`.
- Gửi Quân ảnh dòng "Phiên bản thư viện" ở ô kiểm tra đầu.
- Trên sheet: W4-12 → **Đang review**. Nhắn Quân để đồng bộ lên GitHub.